# 10 · Run controlled experiments and resume training

**Prerequisites:** CNN training, validation, and Python dictionaries.
**Time:** 35–50 minutes. No downloads or earlier notebook artifacts required.

You will compare two learning rates with the same initialization and batch
order, select checkpoints by validation loss, use a scheduler and early
stopping, save experiment records, and actually resume optimizer and random
state. Keep the final test result separate from experiment selection.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import copy
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from course_utils import get_digits_loaders, TinyCNN, evaluate
train_source, val_loader, test_loader = get_digits_loaders(batch_size=64, seed=42)
out_dir = ARTIFACTS / '10_experiments'; out_dir.mkdir(parents=True, exist_ok=True)
max_epochs = 1 if SMOKE_TEST else 8


## 1. Write down the comparison before training

Compare Adam learning rates 0.001 and 0.003; hold all other settings fixed.
We use the same model initialization, split, shuffling seed, batch size,
optimizer, epoch cap, and checkpoint criterion. The chosen result has the
lowest validation cross-entropy. A single seed teaches control but does not
establish statistical superiority; a serious claim needs repeated seeds.


In [ ]:
configs = [dict(name='lr_0p001', lr=0.001, seed=42),
           dict(name='lr_0p003', lr=0.003, seed=42)]
display(pd.DataFrame(configs))

def make_run(config):
    seed_everything(config['seed'])
    model = TinyCNN().to(device)
    generator = torch.Generator().manual_seed(config['seed'])
    loader = DataLoader(train_source.dataset, batch_size=64, shuffle=True,
                        generator=generator, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=1)
    return model, optimizer, scheduler, loader, generator


## 2. Make one training epoch explicit

`model.train()` enables training behavior; `evaluate()` switches to evaluation
behavior. The scheduler receives validation loss **after** each epoch.
Early stopping ends a run after three epochs without meaningful improvement.
With eight epochs it may never trigger; the mechanism is still present.


In [ ]:
def train_epoch(model, optimizer, loader):
    model.train()
    total_loss, total = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.numel()
        total += labels.numel()
    return total_loss / total

def advance_epoch(model, optimizer, scheduler, loader, epoch):
    lr_used = optimizer.param_groups[0]['lr']
    train_loss = train_epoch(model, optimizer, loader)
    result = evaluate(model, val_loader, device=device)
    scheduler.step(result['loss'])
    return {'epoch': epoch, 'lr': lr_used, 'train_loss': train_loss,
            'val_loss': result['loss'], 'val_accuracy': result['accuracy']}


## 3. A resumable checkpoint contains more than weights

We save the model, optimizer momentum, scheduler state, completed epoch,
best validation checkpoint, patience counter, configuration, and history.
We also save Python, NumPy, PyTorch CPU/CUDA, and DataLoader-generator RNG
states. This demo saves **at epoch boundaries** with `num_workers=0`.

Mid-epoch resume additionally requires sampler progress and worker state.
Different devices, PyTorch releases, and nondeterministic GPU kernels can
change numerics; this notebook does not promise bitwise equivalence across
machines. Only load checkpoint files you trust.


In [ ]:
def capture_rng(generator):
    state = np.random.get_state()
    # Built-in values and tensors keep this compatible with weights_only=True.
    return {'python': random.getstate(),
            'numpy': {'name': state[0], 'keys': state[1].tolist(),
                      'pos': state[2], 'has_gauss': state[3], 'cached': state[4]},
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            'loader': generator.get_state()}

def restore_rng(state, generator):
    random.setstate(state['python'])
    n = state['numpy']
    np.random.set_state((n['name'], np.array(n['keys'], dtype=np.uint32),
                         n['pos'], n['has_gauss'], n['cached']))
    torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])
    generator.set_state(state['loader'].cpu())


In [ ]:
def run_experiment(config):
    model, optimizer, scheduler, loader, generator = make_run(config)
    history, best_loss, best_epoch, bad_epochs = [], float('inf'), 0, 0
    best_state = None
    checkpoint_path = out_dir / (config['name'] + '_last.pt')
    for epoch in range(1, max_epochs + 1):
        row = advance_epoch(model, optimizer, scheduler, loader, epoch)
        history.append(row)
        if row['val_loss'] < best_loss - 1e-4:
            best_loss, best_epoch, bad_epochs = row['val_loss'], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1
        torch.save({'config': config, 'epoch': epoch,
                    'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'best_model': best_state,
                    'best_loss': best_loss, 'best_epoch': best_epoch,
                    'bad_epochs': bad_epochs, 'history': history,
                    'rng': capture_rng(generator)}, checkpoint_path)
        if bad_epochs >= 3:
            break
    model.load_state_dict(best_state)
    torch.save(best_state, out_dir / (config['name'] + '_best_weights.pt'))
    return {'config': config, 'model': model, 'history': history,
            'best_loss': best_loss, 'best_epoch': best_epoch,
            'checkpoint_path': checkpoint_path}

experiments = [run_experiment(config) for config in configs]
print('Saved both last-state and best-weight checkpoints.')


## 4. Compare and select using validation only

A best-weight checkpoint is convenient for inference. A last-state checkpoint
is convenient for continuing a interrupted run. They answer different needs:
pairing best weights with a last-epoch optimizer would silently mismatch state.


In [ ]:
summary = pd.DataFrame([{'name': e['config']['name'], 'lr': e['config']['lr'],
                         'seed': e['config']['seed'], 'best_epoch': e['best_epoch'],
                         'best_val_loss': e['best_loss'],
                         'epochs_run': len(e['history'])} for e in experiments])
display(summary.sort_values('best_val_loss'))
chosen = min(experiments, key=lambda e: e['best_loss'])
for e in experiments:
    h = pd.DataFrame(e['history'])
    plt.plot(h['epoch'], h['val_loss'], 'o-', label=e['config']['name'])
plt.xlabel('Epoch'); plt.ylabel('Validation cross-entropy'); plt.legend(); plt.show()
print('Selected before opening test:', chosen['config']['name'])


In [ ]:
summary.to_csv(out_dir / 'experiment_summary.csv', index=False)
details = {'selection_metric': 'lowest validation cross-entropy',
           'torch_version': str(torch.__version__), 'device': str(device),
           'smoke_test': SMOKE_TEST,
           'runs': [{'config': e['config'], 'history': e['history']} for e in experiments]}
(out_dir / 'experiment_details.json').write_text(json.dumps(details, indent=2), encoding='utf-8')
print('Records:', out_dir)


## 5. Restore the last training state and take another epoch

Construct objects first, load their states, then restore RNG state last so
initialization cannot consume the restored random stream. `map_location='cpu'`
loads RNG tensors safely; optimizer loading maps its tensors to parameters.

This demonstration continues the chosen configuration for one additional
epoch. It is a separate validation-only experiment, so it does not silently
replace the already selected best model.


In [ ]:
checkpoint = torch.load(chosen['checkpoint_path'], map_location='cpu', weights_only=True)

def restore_run(checkpoint):
    objects = make_run(checkpoint['config'])
    model, optimizer, scheduler, loader, generator = objects
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    restore_rng(checkpoint['rng'], generator)
    return objects

resumed, resumed_opt, resumed_sched, resumed_loader, resumed_gen = restore_run(checkpoint)
resumed_row = advance_epoch(resumed, resumed_opt, resumed_sched, resumed_loader,
                           checkpoint['epoch'] + 1)
print('Resumed epoch:', resumed_row)
print('Restored scheduler epochs:', resumed_sched.last_epoch)


In [ ]:
# Restore the same snapshot again and compare one-epoch continuation.
replay, replay_opt, replay_sched, replay_loader, replay_gen = restore_run(checkpoint)
replay_row = advance_epoch(replay, replay_opt, replay_sched, replay_loader,
                          checkpoint['epoch'] + 1)
max_difference = max((a.detach() - b.detach()).abs().max().item()
                     for a, b in zip(resumed.parameters(), replay.parameters()))
print('Largest parameter difference after restored continuation:', max_difference)
print('Validation-loss difference:', abs(resumed_row['val_loss'] - replay_row['val_loss']))
print('Zero or tiny differences are expected on a consistent setup; GPU determinism varies.')


## 6. Evaluate the preselected model once

Use the original best checkpoint selected in step 4. The resume demonstration
is not part of test-based model selection. Write the result next to its
configuration so a future reader knows exactly what was measured.


In [ ]:
final = evaluate(chosen['model'], test_loader, device=device)
final_record = {'chosen_config': chosen['config'], 'best_epoch': chosen['best_epoch'],
                'test_loss': final['loss'], 'test_accuracy': final['accuracy']}
(out_dir / 'final_test.json').write_text(json.dumps(final_record, indent=2), encoding='utf-8')
print(final_record)


## Practice 1 · Inspect Adam's hidden state

Compare parameter count with optimizer-state keys. Find the accumulated
first and second moments. Explain why restoring only weights is not the
same as continuing training with Adam.


In [ ]:
first_state = next(iter(checkpoint['optimizer']['state'].values()))
print('One parameter optimizer state:', {key: tuple(value.shape) if torch.is_tensor(value) else value
                                         for key, value in first_state.items()})
print('Model tensors:', len(checkpoint['model']))
print('Optimizer parameter entries:', len(checkpoint['optimizer']['state']))


<details><summary>Solution idea</summary>
Adam maintains a step count, exp_avg (first moment), and exp_avg_sq (second
moment). Resetting them changes the update even when the model weights match.
</details>

## Practice 2 · Observe patience and scheduling without another model

Feed a synthetic validation-loss sequence to a fresh scheduler. Change its
patience and predict when the learning rate will drop. The tiny optimizer
steps below preserve the normal optimizer-before-scheduler order.


In [ ]:
toy_parameter = nn.Parameter(torch.tensor(1.0))
toy_optimizer = torch.optim.SGD([toy_parameter], lr=0.1)
toy_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(toy_optimizer, patience=1, factor=0.5)
for observed_loss in [1.0, 0.8, 0.81, 0.82, 0.83, 0.7]:
    toy_optimizer.zero_grad(); (toy_parameter ** 2).backward(); toy_optimizer.step()
    toy_scheduler.step(observed_loss)
    print('Loss:', observed_loss, '| next learning rate:', toy_optimizer.param_groups[0]['lr'])


<details><summary>Solution idea</summary>
With patience 1, the scheduler tolerates one bad epoch and reduces the rate
after the next one. Early stopping has its own counter; choose its patience
long enough to allow a learning-rate reduction a chance to help.
</details>

## Practice 3 · Design a repeated-seed comparison

Create the experiment plan below. Set `RUN_REPEATED_SEEDS=True` to run it
and summarize validation losses. It never reads the test set. The optional
runs use unique filenames, preserving the initial experiment records.


In [ ]:
repeated_plan = [dict(name=f'repeat_lr{lr}_seed{seed}', lr=lr, seed=seed)
                 for seed in [10, 20, 30] for lr in [0.001, 0.003]]
display(pd.DataFrame(repeated_plan))
RUN_REPEATED_SEEDS = False  # Change to True when ready for six short runs.
if RUN_REPEATED_SEEDS and not SMOKE_TEST:
    repeated = [run_experiment(config) for config in repeated_plan]
    table = pd.DataFrame([{'lr': e['config']['lr'], 'seed': e['config']['seed'],
                           'val_loss': e['best_loss']} for e in repeated])
    display(table.groupby('lr')['val_loss'].agg(['mean', 'std', 'count']))
else:
    print('Plan ready. Enable the switch to estimate variation across seeds.')


<details><summary>Solution idea</summary>
Pair learning rates within each seed and compare per-seed differences.
Report variation and compute cost; a tiny mean difference may not be robust.
Repeated searches can overfit validation, so a final fresh holdout matters.
</details>

**Completion check:** identify your selection metric, distinguish last versus
best checkpoints, restore optimizer/scheduler/RNG state, and trace a result
from the JSON/CSV files back to its configuration.

**Primary sources:** [PyTorch saving and loading](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html),
[reproducibility notes](https://docs.pytorch.org/docs/2.11/notes/randomness.html),
[ReduceLROnPlateau](https://docs.pytorch.org/docs/2.11/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html).

**Next:** [11 · Mixed precision, mixup, and distillation](11_mixed_precision_mixup_distillation.ipynb).
